In [1]:
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import \
    metrics_search_for_two_fragments_df
import pvlib
import pytz
from datetime import date, datetime, timedelta

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records', 'sample_year'],
      dtype='object')

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [5]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [6]:
num_good_data_sites = len(enough_data_parquet_power_list)

In [8]:
compare_locations = pd.DataFrame(
    data = np.zeros((num_good_data_sites, 4), dtype='float64'),
    index = enough_data_parquet_power_list,
    columns = ['actual_lat', 'actual_long', 'approx_lat', 'approx_long']
)
for system_id in enough_data_parquet_power_list:
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    compare_locations.at[system_id, 'actual_lat'] = relevant_rows_systems.at[first_ind, 'latitude']
    compare_locations.at[system_id, 'actual_long'] = relevant_rows_systems.at[first_ind, 'longitude']
    # made a mistake with metadata, but works for us here
    metadata_path = f'../../../../data_ds_project/nsrdb/{system_id}/metadata/year.json'
    with open(metadata_path, 'r') as reader:
        term_dict = json.load(reader)
        compare_locations.at[system_id, 'approx_lat'] = term_dict['latitude']
        compare_locations.at[system_id, 'approx_long'] = term_dict['longitude']

In [9]:
compare_locations

,actual_lat,actual_long,approx_lat,approx_long
4,39.7406,-105.1774,39.73,-105.18
10,39.7404,-105.1774,39.73,-105.18
33,39.7404,-105.1772,39.73,-105.18
34,36.1952,-115.1582,36.21,-115.14
35,36.1952,-115.1582,36.21,-115.14
...,...,...,...,...
1432,39.7422,-105.1719,39.73,-105.18
1433,39.7404,-105.1719,39.73,-105.18
4901,39.1385,-77.2155,39.13,-77.22
4902,39.1319,-77.2141,39.13,-77.22


In [10]:
compare_locations[['approx_lat', 'approx_long']].value_counts()

approx_lat  approx_long
39.73       -105.18        13
29.97       -90.02          9
41.01       -73.66          6
30.01       -90.10          6
27.77       -82.66          5
36.21       -115.14         5
29.97       -90.10          5
39.13       -77.22          3
39.93       -75.06          2
27.77       -82.62          2
36.05       -114.94         2
27.81       -82.70          2
29.93       -90.14          2
29.97       -90.06          1
39.49       -76.66          1
44.45       -73.10          1
27.81       -82.66          1
39.89       -105.22         1
39.81       -75.38          1
39.73       -75.54          1
27.81       -82.62          1
39.69       -105.18         1
39.21       -76.70          1
29.97       -90.14          1
28.41       -80.78          1
29.01       -80.94          1
36.17       -115.02         1
            -115.14         1
36.13       -115.26         1
36.05       -114.98         1
36.01       -114.94         1
30.01       -90.06          1
46.69       -68.

In [11]:
compare_locations[['approx_lat', 'approx_long']].nunique()

approx_lat     23
approx_long    25
dtype: int64

In [13]:
compare_locations_sort = compare_locations.sort_values('approx_long')
compare_locations_sort.head()

,actual_lat,actual_long,approx_lat,approx_long
1418,36.1253,-115.2713,36.13,-115.26
1276,36.1952,-115.1582,36.21,-115.14
1277,36.1952,-115.1582,36.21,-115.14
34,36.1952,-115.1582,36.21,-115.14
35,36.1952,-115.1582,36.21,-115.14


In [16]:
compare_locations.index.name = 'system_id'
compare_locations_no_index = compare_locations.reset_index()
compare_locations_groups = compare_locations_no_index.groupby(['approx_lat', 'approx_long'])['system_id'].unique()
compare_locations_groups

approx_lat  approx_long
27.77       -82.66                            [1310, 1318, 1321, 1325, 1329]
            -82.62                                              [1300, 1319]
27.81       -82.70                                              [1307, 1326]
            -82.66                                                    [1308]
            -82.62                                                    [1314]
28.41       -80.78                                                    [1403]
29.01       -80.94                                                    [1231]
29.93       -90.14                                              [1256, 1273]
29.97       -90.14                                                    [1249]
            -90.10                            [1248, 1250, 1252, 1253, 1272]
            -90.06                                                    [1275]
            -90.02         [1261, 1263, 1264, 1265, 1266, 1268, 1269, 127...
30.01       -90.10                      [1244, 1246,